# 论文 12：量子化学的神经消息传递
## Justin Gilmer, Samuel S. Schoenholz, Patrick F. Riley, Oriol Vinyals, George E. Dahl（2017）

### 消息传递神经网络 (MPNN)

图神经网络的统一框架。现代 GNN 的基础！

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import networkx as nx

np.random.seed(42)

## 图表示

In [ ]:
class Graph:
    '简单的图表示。'
    def __init__(self, num_nodes):
        self.num_nodes = num_nodes
        self.edges = []  # （源、目标）元组列表
        self.node_features = []  # 节点特征向量列表
        self.edge_features = {}  # 字典：(src, tgt) -> 边特征
    
    def add_edge(self, src, tgt, features=None):
        self.edges.append((src, tgt))
        if features is not None:
            self.edge_features[(src, tgt)] = features
    
    def set_node_features(self, features):
        'features：节点特征向量列表。'
        self.node_features = features
    
    def get_neighbors(self, node):
        '获取节点的所有邻居'
        neighbors = []
        for src, tgt in self.edges:
            if src == node:
                neighbors.append(tgt)
        return neighbors
    
    def visualize(self, node_labels=None):
        '使用 Networkx 可视化图'
        G = nx.DiGraph()
        G.add_nodes_from(range(self.num_nodes))
        G.add_edges_from(self.edges)
        
        pos = nx.spring_layout(G, seed=42)
        
        plt.figure(figsize=(10, 8))
        nx.draw(G, pos, with_labels=True, node_color='lightblue', 
               node_size=800, font_size=12, arrows=True,
               arrowsize=20, edge_color='gray', width=2)
        
        if node_labels:
            nx.draw_networkx_labels(G, pos, node_labels, font_size=10)
        
        plt.title("Graph Structure")
        plt.axis('off')
        plt.show()

# 创建样本分子图
# H2O（水）：O 连接到 2 个 H 原子
water = Graph(num_nodes=3)
water.add_edge(0, 1)  # O -> H
water.add_edge(0, 2)  # O -> H  
water.add_edge(1, 0)  # H -> O（无向）
water.add_edge(2, 0)  # H -> O

# 节点特征：[atomic_num, valence, ...]
water.set_node_features([
    np.array([8, 2]),  # 氧
    np.array([1, 1]),  # 氢
    np.array([1, 1]),  # 氢
])

labels = {0: 'O', 1: 'H', 2: 'H'}
water.visualize(labels)

print(f"Number of nodes: {water.num_nodes}")
print(f"Number of edges: {len(water.edges)}")
print(f"Neighbors of node 0 (Oxygen): {water.get_neighbors(0)}")

## 消息传递框架

**两个阶段：**
1. **消息传递**：聚合来自邻居的信息（T 步骤）
2. **读出**：生成全局图表示

$$m_v^{t+1} = \sum_{w \in N(v)} M_t(h_v^t, h_w^t, e_{vw})$$
$$h_v^{t+1} = U_t(h_v^t, m_v^{t+1})$$

In [ ]:
class MessagePassingLayer:
    '单层消息传递。'
    def __init__(self, node_dim, edge_dim, hidden_dim):
        self.node_dim = node_dim
        self.edge_dim = edge_dim
        self.hidden_dim = hidden_dim
        
        # 消息函数：M(h_v, h_w, e_vw)
        self.W_msg = np.random.randn(hidden_dim, 2*node_dim + edge_dim) * 0.01
        self.b_msg = np.zeros(hidden_dim)
        
        # 更新函数：U(h_v, m_v)
        self.W_update = np.random.randn(node_dim, node_dim + hidden_dim) * 0.01
        self.b_update = np.zeros(node_dim)
    
    def message(self, h_source, h_target, e_features):
        '计算从源到目标的消息'
        # 拼接源节点、目标节点和边特征
        if e_features is None:
            e_features = np.zeros(self.edge_dim)
        
        concat = np.concatenate([h_source, h_target, e_features])
        
        # 应用消息网络
        message = np.tanh(np.dot(self.W_msg, concat) + self.b_msg)
        return message
    
    def aggregate(self, messages):
        '聚合消息（总和）'
        if len(messages) == 0:
            return np.zeros(self.hidden_dim)
        return np.sum(messages, axis=0)
    
    def update(self, h_node, aggregated_message):
        '更新节点表示'
        concat = np.concatenate([h_node, aggregated_message])
        h_new = np.tanh(np.dot(self.W_update, concat) + self.b_update)
        return h_new
    
    def forward(self, graph, node_states):
        """一步消息传递
        
        graph：图对象
        node_states：当前节点隐藏状态列表
        
        返回：更新的节点状态"""
        new_states = []
        
        for v in range(graph.num_nodes):
            # 收集邻居的消息
            messages = []
            for w in graph.get_neighbors(v):
                # 获取边特征
                edge_feat = graph.edge_features.get((w, v), None)
                
                # 计算消息
                msg = self.message(node_states[w], node_states[v], edge_feat)
                messages.append(msg)
            
            # 聚合消息
            aggregated = self.aggregate(messages)
            
            # 更新节点状态
            h_new = self.update(node_states[v], aggregated)
            new_states.append(h_new)
        
        return new_states

# 测试消息传递
node_dim = 4
edge_dim = 2
hidden_dim = 8

mp_layer = MessagePassingLayer(node_dim, edge_dim, hidden_dim)

# 从特征初始化节点状态
initial_states = []
for feat in water.node_features:
    # 嵌入到更高维度
    state = np.concatenate([feat, np.zeros(node_dim - len(feat))])
    initial_states.append(state)

# 运行消息传递
updated_states = mp_layer.forward(water, initial_states)

print(f"\nInitial state (O): {initial_states[0]}")
print(f"Updated state (O): {updated_states[0]}")
print(f"\nNode states updated via neighbor information!")

## 完整的 MPNN

In [ ]:
class MPNN:
    '消息传递神经网络。'
    def __init__(self, node_feat_dim, edge_feat_dim, hidden_dim, num_layers, output_dim):
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers
        
        # 嵌入层
        self.embed_W = np.random.randn(hidden_dim, node_feat_dim) * 0.01
        
        # 消息传递层
        self.mp_layers = [
            MessagePassingLayer(hidden_dim, edge_feat_dim, hidden_dim*2)
            for _ in range(num_layers)
        ]
        
        # 读出（图级预测）
        self.readout_W = np.random.randn(output_dim, hidden_dim) * 0.01
        self.readout_b = np.zeros(output_dim)
    
    def forward(self, graph):
        """通过 MPNN 进行前向传播
        
        返回：图级预测"""
        # 嵌入节点特征
        node_states = []
        for feat in graph.node_features:
            embedded = np.tanh(np.dot(self.embed_W, feat))
            node_states.append(embedded)
        
        # 消息传递
        states_history = [node_states]
        for layer in self.mp_layers:
            node_states = layer.forward(graph, node_states)
            states_history.append(node_states)
        
        # 读出：将节点状态聚合为图表示
        graph_repr = np.sum(node_states, axis=0)  # 简单的总和池
        
        # 最终预测
        output = np.dot(self.readout_W, graph_repr) + self.readout_b
        
        return output, states_history

# 创建MPNN
mpnn = MPNN(
    node_feat_dim=2,
    edge_feat_dim=2,
    hidden_dim=8,
    num_layers=3,
    output_dim=1  # 预测单个属性，例如能量
)

# 前向传播
prediction, history = mpnn.forward(water)

print(f"Graph-level prediction: {prediction}")
print(f"(E.g., molecular property like energy, solubility, etc.)")

## 可视化消息传递

In [ ]:
# 可视化节点表示如何演变
fig, axes = plt.subplots(1, len(history), figsize=(16, 4))

for step, states in enumerate(history):
    # 用于可视化的堆栈节点状态
    states_matrix = np.array(states).T  # （hidden_dim、num_nodes）
    
    ax = axes[step]
    im = ax.imshow(states_matrix, cmap='RdBu', aspect='auto')
    ax.set_title(f'Step {step}')
    ax.set_xlabel('Node')
    ax.set_ylabel('Hidden Dimension')
    ax.set_xticks([0, 1, 2])
    ax.set_xticklabels(['O', 'H', 'H'])

plt.colorbar(im, ax=axes, label='Activation')
plt.suptitle('Node Representations Through Message Passing', fontsize=14)
plt.tight_layout()
plt.show()

print("\nNodes update their representations by aggregating neighbor information")

## 创建更复杂的图

In [ ]:
# 创建苯环（C6H6）
benzene = Graph(num_nodes=12)  # 6 C + 6 H

# 碳环（节点0-5）
for i in range(6):
    next_i = (i + 1) % 6
    benzene.add_edge(i, next_i)
    benzene.add_edge(next_i, i)

# 氢原子（节点 6-11）附着在碳上
for i in range(6):
    h_idx = 6 + i
    benzene.add_edge(i, h_idx)
    benzene.add_edge(h_idx, i)

# 节点特征
features = []
for i in range(6):
    features.append(np.array([6, 3]))  # 碳
for i in range(6):
    features.append(np.array([1, 1]))  # 氢
benzene.set_node_features(features)

# 可视化
labels = {i: 'C' for i in range(6)}
labels.update({i: 'H' for i in range(6, 12)})
benzene.visualize(labels)

# 运行 MPNN
pred_benzene, hist_benzene = mpnn.forward(benzene)
print(f"\nBenzene prediction: {pred_benzene}")

## 不同的聚合函数

In [ ]:
# 比较聚合策略
def sum_aggregation(messages):
    return np.sum(messages, axis=0) if len(messages) > 0 else np.zeros_like(messages[0])

def mean_aggregation(messages):
    return np.mean(messages, axis=0) if len(messages) > 0 else np.zeros_like(messages[0])

def max_aggregation(messages):
    return np.max(messages, axis=0) if len(messages) > 0 else np.zeros_like(messages[0])

# 测试随机消息
test_messages = [np.random.randn(8) for _ in range(3)]

print("Aggregation Functions:")
print(f"Sum: {sum_aggregation(test_messages)[:4]}...")
print(f"Mean: {mean_aggregation(test_messages)[:4]}...")
print(f"Max: {max_aggregation(test_messages)[:4]}...")
print("\nDifferent aggregations capture different patterns!")

## 要点

### 消息传递框架：

**阶段 1：消息传递**（重复 T 次）
```
For each node v:
  1. Collect messages from neighbors:
     m_v = Σ_{u∈N(v)} M_t(h_v, h_u, e_uv)
  
  2. Update node state:
     h_v = U_t(h_v, m_v)
```

**阶段 2：读出**
```
Graph representation:
  h_G = R({h_v | v ∈ G})
```

### 组成部分
1. **消息函数 M**：计算来自邻居的消息
2. **聚合**：组合消息（总和、平均值、最大值、注意力）
3. **更新函数 U**：更新节点表示
4. **读出 R**：图级池化

### 变体
- **GCN**：通过标准化简化消息传递
- **GraphSAGE**：邻居采样，归纳学习
- **GAT**：基于注意力的聚合
- **GIN**：具有更强表达能力的聚合（sum + MLP）

### 应用：
- **分子属性预测**：QM9、药物发现
- **社交网络**：节点分类、链接预测
- **知识图谱**：推理、补全
- **推荐系统**：用户-物品图
- **3D 视觉**：点云、网格

### 优点：
- ✅ 处理大小可变的图
- ✅ 具有置换不变性
- ✅ 支持归纳学习，可以泛化到新图
- ✅ 可解释（消息传递）

### 挑战：
- 过度平滑（深层使节点相似）
- 表现力（受聚合限制）
- 可扩展性（大图）

### 现代扩展：
- **图 Transformer**：在完整图结构上应用注意力
- **等变 GNN**：尊重对称性（E(3)、SE(3)）
- **时态 GNN**：动态图
- **异构 GNN**：处理多种节点类型和边类型